# **Language Ablation**

idk what to write here yet lols

In [1]:
import csv
import pandas as pd
import os
from pathlib import Path
import glob

import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler

import sys
parent_dir = str(Path.cwd().parent)
sys.path.append(parent_dir)

from src.prep_data import *
from src.decode import *
from src.train import *
from src.temperature import *

## Cleaning & Organizing 

Taking Wikipron data and turning it into dataframes + removing duplicate,null rows

In [2]:
input_dir_ph = Path("./wikipron/filipino")
input_dir_austro = Path("./wikipron/austronesian")

ph_languages = ["bikolano", "cebuano", "hiligaynon", "ilocano", "kapampangan", "pangasinan", "tagalog", "waray"]
austro_languages = ["acehnese", "balinese", "iban", "indonesian", "makasar", "malay", "west makian"]
all_languages = ph_languages + austro_languages + ["tagalog"]
col_names = ["text", "phonetic", "language", "group"]

In [3]:
ph_files = glob.glob("../wikipron/filipino/*.tsv")
austro_files = glob.glob("../wikipron/austronesian/*.tsv")

In [4]:
temp_ph = []
temp_austro = []

for filename in ph_files:
    temp_df = pd.read_csv(filename, sep='\t', names=col_names, header=None)
    temp_df["language"] = Path(filename).stem
    temp_df["group"] = "philippine"
    temp_ph.append(temp_df)

for filename in austro_files:
    temp_df = pd.read_csv(filename, sep='\t', names=col_names, header=None)
    temp_df["language"] = Path(filename).stem
    temp_df["group"] = "austronesian"
    temp_austro.append(temp_df)

ph_df = pd.concat(temp_ph, ignore_index=True)
austro_df = pd.concat(temp_austro, ignore_index=True)
all_df = pd.concat(temp_austro + temp_ph, ignore_index=True)

In [5]:
print(ph_df.head())
print("\n")
print(austro_df.head())

print("\n")
print(ph_df["language"].unique())
print("\n")
print(austro_df["language"].unique())
print("\n")
print(all_df["language"].unique())

      text       phonetic  language       group
0    Abril    ʔ a b ɾ i l  bikolano  philippine
1   Agosto  ʔ a ɡ o s t o  bikolano  philippine
2  Aguilar  ʔ a ɡ i l a ɾ  bikolano  philippine
3    Albay    ʔ a l b a j  bikolano  philippine
4   Alzaga  ʔ a l s a ɡ a  bikolano  philippine


      text      phonetic  language         group
0     Acèh       a c ɛ h  acehnese  austronesian
1  Aleuhat  a l ɯ h a t̠  acehnese  austronesian
2    abang       a b a ŋ  acehnese  austronesian
3     adèe      a d ɛ ə̯  acehnese  austronesian
4    aleue      a l ɯ ə̯  acehnese  austronesian


<StringArray>
[   'bikolano',     'cebuano',  'hiligaynon',     'ilocano', 'kapampangan',
  'pangasinan',       'waray']
Length: 7, dtype: str


<StringArray>
['acehnese', 'balinese', 'iban', 'indonesian', 'makasar', 'malay',
 'west_makian']
Length: 7, dtype: str


<StringArray>
[   'acehnese',    'balinese',        'iban',  'indonesian',     'makasar',
       'malay', 'west_makian',    'bikolano',     'cebuano

## Splitting

In [6]:
input_dir_tgl = Path("../wikipron/tagalog.tsv")

df_tgl = pd.read_csv(input_dir_tgl, sep='\t', names=col_names, header=None)
df_tgl["language"] = "tagalog"
df_tgl["group"] = "philippine"

In [7]:
from sklearn.model_selection import train_test_split

train_dev_tgl, test_tgl = train_test_split(df_tgl, test_size=0.15, random_state=42)
train_tgl, dev_tgl = train_test_split(train_dev_tgl, test_size=0.1765, random_state=42)

ph_aux_df = ph_df.copy()
austro_aux_df = austro_df.copy()

tagalog_only_df = train_tgl.copy()
ph_df = pd.concat([train_tgl, ph_aux_df], ignore_index=True)
austro_df = pd.concat([train_tgl, austro_aux_df], ignore_index=True)
all_df = pd.concat([train_tgl, ph_aux_df, austro_aux_df], ignore_index=True)

for d in [tagalog_only_df, ph_df, austro_df, all_df]:
    d.drop_duplicates(subset=["text", "phonetic"], inplace=True)
    d.dropna(inplace=True)

In [8]:
# ph_df.to_csv("ablation/input/ph_train.tsv", sep="\t", index=False, header=False)
# austro_df.to_csv("ablation/input/austro_train.tsv", sep="\t", index=False, header=False)
# all_df.to_csv("ablation/input/all_train.tsv", sep="\t", index=False, header=False)
# dev_tgl.to_csv("ablation/input/tgl_dev.tsv", sep="\t", index=False, header=False)
# test_tgl.to_csv("ablation/input/tgl_test.tsv", sep="\t", index=False, header=False)

In [9]:
input_dir = Path("ablation/input")
output_dir = Path("ablation/splits")
output_dir.mkdir(parents=True, exist_ok=True)

combined_df = pd.concat(
    [ph_aux_df, austro_aux_df, train_tgl], ignore_index=True
).drop_duplicates(subset=["text", "phonetic"])

for lang, group_df in combined_df.groupby("language"):
    group_df[["text", "phonetic"]].to_csv(
        input_dir / f"{lang}_train.tsv", sep="\t", index=False, header=False
    )
    print(f"{lang}: {len(group_df)} pairs")

dev_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_dev.tsv", sep="\t", index=False, header=False)
test_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_test.tsv", sep="\t", index=False, header=False)


acehnese: 265 pairs
balinese: 295 pairs
bikolano: 5432 pairs
cebuano: 3031 pairs
hiligaynon: 165 pairs
iban: 552 pairs
ilocano: 827 pairs
indonesian: 18199 pairs
kapampangan: 900 pairs
makasar: 766 pairs
malay: 4323 pairs
pangasinan: 158 pairs
tagalog: 17220 pairs
waray: 186 pairs
west_makian: 740 pairs


In [10]:
sets = {
    "ph": ph_languages,
    "austro": austro_languages,
    "all": all_languages,
}

for set_name, langs in sets.items():
    out_dir = f"ablation/splits/{set_name}"
    print(f"Set: {set_name}")
    convert_split(input_dir, out_dir, langs, "train")

for split in ["dev", "test"]:
    out_dir = f"ablation/splits/tgl_{split}"
    print(f"Split: {split}")
    convert_split(input_dir, out_dir, ["tagalog"], split)

Set: ph
c:\Users\Erin\Documents\GITHUB\Hunger\notebooks
  bikolano train: 5432 pairs
  cebuano train: 3031 pairs
  hiligaynon train: 165 pairs
  ilocano train: 827 pairs
  kapampangan train: 900 pairs
  pangasinan train: 158 pairs
  tagalog train: 17220 pairs
  waray train: 186 pairs
  -> wrote 27919 lines to train.{src,tgt,lang}
Set: austro
c:\Users\Erin\Documents\GITHUB\Hunger\notebooks
  acehnese train: 265 pairs
  balinese train: 295 pairs
  iban train: 552 pairs
  indonesian train: 18199 pairs
  makasar train: 766 pairs
  malay train: 4323 pairs
  [skip] ablation\input\west makian_train.tsv not found
  -> wrote 24400 lines to train.{src,tgt,lang}
Set: all
c:\Users\Erin\Documents\GITHUB\Hunger\notebooks
  bikolano train: 5432 pairs
  cebuano train: 3031 pairs
  hiligaynon train: 165 pairs
  ilocano train: 827 pairs
  kapampangan train: 900 pairs
  pangasinan train: 158 pairs
  tagalog train: 17220 pairs
  waray train: 186 pairs
  acehnese train: 265 pairs
  balinese train: 295 pair

## Train

In [11]:
DATA = Path("ablation/splits")
OUTPUT = Path("./models")
#seed is a var
STEPS = 1000
SAVE_STEP = 1000

BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

# if len(xm.get_xla_supported_devices()) > 0:
#     DEVICE = "xla"
if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

xla = DEVICE == "xla"

print(DEVICE)

#loging is a var

CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 100

cuda


In [ ]:
sets = ["ph", "austro", "all"]
seed = 42

if xla:

    import torch_xla.distributed.parallel_loader as pl
    device = xm.xla_device()
    torch.manual_seed(seed)  # xm also seeds RNG on optimizer_step
else:
    device = torch.device(DEVICE)
    torch.manual_seed(seed)

for set_name in sets:
    print(f"\nTraining set: {set_name}")
    DATA = Path(f"ablation/splits/{set_name}")
    OUTPUT = Path(f"./models/{set_name}")
    os.makedirs(OUTPUT, exist_ok=True)

    config_1 = dict(CONFIG)
    config_1["seed"] = seed
    config_1["data_path"] = str(DATA)
    config_1["output_path"] = str(OUTPUT)

    torch.manual_seed(seed)
    device = torch.device(DEVICE)

    src_vocab, tgt_vocab = build_vocabs(
        os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
    save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

    train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                            os.path.join(DATA, "train.tgt"),
                            src_vocab, tgt_vocab)
    max_src, max_tgt = compute_max_lengths(train_ds)
    print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
            f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

    weights = temperature_sampling_weights(
        os.path.join(DATA, "train.lang"), temperature=5.0
    )
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                        collate_fn=make_fixed_collate(max_src, max_tgt),
                        drop_last=True)
    if xla:
        loader = pl.MpDeviceLoader(loader, device)  # preloads batches onto the TPU

    model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                        n_heads=NUM_HEADS, d_ff=FF_SIZE,
                        n_layers=N_LAYERS, dropout=DROPOUT,
                        pad_id=PAD_ID).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
    sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
    criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

    model.train()
    data_iter = infinite_loader(loader)
    for step in range(1, STEPS + 1):
        batch = next(data_iter)
        if xla:
            src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
        else:
            src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

        logits = model(src, tgt_in, src_pad, tgt_pad)
        loss = criterion(logits, tgt_out)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

        if step % log_step == 0:
            # .item() forces a host sync; only do it at log intervals on TPU.
            print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
                    f"lr {opt.param_groups[0]['lr']:.2e}")

        if step % SAVE_STEP == 0 or step == STEPS:
            ckpt = os.path.join(OUTPUT, f"ckpt_{step}.pt")
            payload = {"model": model.state_dict(),
                        "config": config_1,
                        "src_vocab": src_vocab.to_dict(),
                        "tgt_vocab": tgt_vocab.to_dict()}
            if xla:
                xm.save(payload, ckpt)   # moves tensors to CPU; master-only
            else:
                torch.save(payload, ckpt)
            print(f"  saved {ckpt}")
        


=== Training condition: ph ===
src vocab=74 tgt vocab=67 examples=27919 | fixed shapes: src=48 tgt=32
Language distribution (natural -> temperature-scaled):
  bikolano     n=  5432  p=0.1946  q=0.1658
  cebuano      n=  3031  p=0.1086  q=0.1475
  hiligaynon   n=   165  p=0.0059  q=0.0824
  ilocano      n=   827  p=0.0296  q=0.1138
  kapampangan  n=   900  p=0.0322  q=0.1157
  pangasinan   n=   158  p=0.0057  q=0.0817
  tagalog      n= 17220  p=0.6168  q=0.2088
  waray        n=   186  p=0.0067  q=0.0844


## Evaluation